# Fine-tuning Moonshine-Base-BN on Bangla conversational + phone-style speech

**Goal of this notebook:** check whether fine-tuning `Sanjidh090/moonshine-base-bn` on *open* conversational Bangla
speech (plus simulated phone-line audio) makes it better on call-center-like audio, **before** we spend time
labelling our own call recordings.

**Runs on:** Google Colab, free **T4 GPU** (Runtime → Change runtime type → T4 GPU).

| Part | What happens | Time on a T4 (rough) |
|---|---|---|
| 1 | Read: how ASR training works (no code) | 10 min reading |
| 2 | Setup: install, Google Drive, Hugging Face login | 3 min |
| 3 | Load the model, look at tokens and labels | 1 min |
| 4 | Download + inspect + prepare the data | 15–30 min |
| 5 | Phone-call simulation (listen to it!) | 1 min |
| 6 | **Baseline**: score the original model *before* training | 5–10 min |
| 7 | **Train** | 30–90 min (depends on hours of data) |
| 8 | **Evaluate**: before vs. after, on every test set + your own call recordings | 10 min |
| 9 | Save the model and use it on your PC | 2 min |

> **First run `QUICK_TEST = True`** (in the CONFIG cell). It runs every step with tiny data in ~10 minutes, so
> you find setup problems (login, dataset access, disk) before the long run. Then set it to `False` and
> *Runtime → Restart and run all*.

The full plan, the dataset choice and the reasons behind it are in `README.md` next to this notebook.

---
# Part 1: How training a speech-to-text model works (read this once)

### 1.1 What Moonshine is

Moonshine is an **encoder–decoder (sequence-to-sequence) transformer**, like Whisper but smaller (61.5M parameters).

```
 16 kHz waveform ──► [ conv stem ] ──► [ ENCODER: 8 transformer layers ] ──► audio features (~41 per second)
                                                                                  │ (cross-attention)
 [CLS] আমি  ভাত  ──► [ DECODER: 8 transformer layers ] ──► next-token probabilities ─► আমি ভাত খাই [SEP]
```

* The **encoder** turns raw audio into a sequence of feature vectors (it "listens").
* The **decoder** writes the text **one token at a time**. Each new token is predicted from (a) the audio
  features and (b) the tokens it already wrote. This is called *autoregressive* decoding.
* Tokens come from the **BanglaBERT WordPiece vocabulary** (32,000 entries). Common words are one token
  (`হাসপাতাল`), rare words are split into pieces (`রেজা` + `##উল`). `[CLS]` (id 2) = "start writing",
  `[SEP]` (id 3) = "I'm done", `[PAD]` (id 0) = filler.

### 1.2 What "training" means here

We show the model `(audio, correct text)` pairs. For each pair:

1. **Labels** = the correct text as token ids + `[SEP]`: `[আমি, ভাত, খাই, SEP]`
2. **Decoder input** = the labels shifted one step right, with `[CLS]` in front: `[CLS, আমি, ভাত, খাই]`
   (the model code does this shift itself, we only give it the labels).
3. At every position the decoder must predict the *next* label token. Because it always sees the **correct**
   previous tokens (not its own guesses), this is called **teacher forcing**. It makes training fast and stable.
4. **Loss** = cross-entropy = how surprised the model was by each correct token, averaged. Lower is better.
5. **Backpropagation** computes how each of the 61.5M weights should change to lower the loss, and the
   **optimizer (AdamW)** moves every weight a tiny step in that direction. The step size is the **learning rate**.

One **step** = one batch of clips → loss → weight update. One **epoch** = every training clip seen once.

### 1.3 Training from scratch vs. fine-tuning

The model already learned Bangla from ~880 hours of speech (the Lipi-Ghor dataset). **Fine-tuning** = continue
training that same model on new data with a **small learning rate** (here `3e-5`), so it *adapts* (phone
sound, conversational style) instead of *re-learning* everything. Two risks:

| Risk | What it looks like | What we do against it |
|---|---|---|
| **Catastrophic forgetting** | Gets better on phone audio but worse on clean audio | Low learning rate, few epochs, mix ~20% clean speech into training, **measure** clean audio before/after |
| **Overfitting** | Training loss keeps going down, dev WER goes *up* | Watch dev WER every few hundred steps, keep the best checkpoint (not the last) |

### 1.4 How we measure: WER and CER

* **WER** (word error rate) = (substituted + deleted + inserted words) / words in the reference.
  `0.30` = 30% of words wrong. Lower is better. Can be > 1 if the model invents many words.
* **CER** = the same on characters. For Bangla, CER is often more informative: `হাসপাতাল` vs `হাসপাতালে`
  is a full word error but only 1 character error.
* Loss is **not** the same as WER. Low loss with teacher forcing does not guarantee good free-running output
  (the model card of this model talks about exactly this), so we evaluate by actually **generating text**.

### 1.5 Train / dev / test: three separate sets

| Set | Used for | Rule |
|---|---|---|
| **train** | updating the weights | the only data the model learns from |
| **dev** (validation) | choosing *when to stop* and which checkpoint is best | never trained on, but we look at it often |
| **test** | the final honest score | never trained on, never used to choose anything |

If test audio leaks into training, the score is fake. That's why our test sets come from **different datasets**
(BanSpeech) than the training data (IndicVoices + SUBAK.KO).

### 1.6 Why "phone simulation"

Our calls are **8 kHz** phone audio: everything above ~3.4 kHz is gone, and the phone codec adds distortion.
The model learned on wide-band YouTube audio. We can turn any clean clip into phone-like audio
(downsample to 8 kHz, band-pass 300–3400 Hz, G.711 µ-law codec, line noise). Training on that teaches the model
the phone sound, and testing on it shows the problem: on the original model, phone-simulated talk-show clips
were ~**17 WER points worse** than the same clips clean (35% → 52% WER in our check).

### 1.7 Words you'll see in the training log

| Term | Meaning |
|---|---|
| **batch size** | clips processed together in one forward pass (16 here) |
| **gradient accumulation** | add up gradients of N batches before one update → effective batch = 16 × 2 = 32, without needing more GPU memory |
| **warm-up** | first 10% of steps the learning rate rises from 0 to its max, so the first noisy updates don't damage the model |
| **linear decay** | after warm-up, learning rate goes down to 0 by the end |
| **fp16 / mixed precision** | most maths in 16-bit floats: ~2× faster on a T4 and less memory. The T4 has no bf16. |
| **weight decay** | small pull of weights towards 0, a mild protection against overfitting |
| **grad norm** | size of the update. Sudden huge values or `nan` = unstable training → lower the learning rate |
| **group by length** | batch clips of similar length together → less padding → faster |

---
# Part 2: Setup

**Before running:**
1. *Runtime → Change runtime type → T4 GPU*.
2. On Hugging Face, open https://huggingface.co/datasets/ai4bharat/IndicVoices, log in, and **accept the
   access conditions** (it's free; the dataset is gated). Without this the download fails.
3. Create a Hugging Face **read token** (https://huggingface.co/settings/tokens) and add it to Colab:
   🔑 icon in the left sidebar → *Add new secret* → name `HF_TOKEN`, value = your token, enable notebook access.
4. (Optional) upload your test recordings (`sample-audio` folder from the project) to Google Drive at
   `MyDrive/moonshine_finetune/sample-audio/`. Section 8 transcribes them with the old and the new model.

In [ ]:
# Is there a GPU? You should see "Tesla T4" below. If not: Runtime -> Change runtime type -> T4 GPU.
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Pin transformers to the version this notebook was tested with (the model needs transformers 5.x).
!pip install -q "transformers==5.22.2" "accelerate>=1.10" "datasets>=4.0" jiwer soundfile "librosa>=0.10" huggingface_hub

In [ ]:
import sys, os
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    # Google Drive survives a Colab disconnect; /content does not. Checkpoints + results go to Drive.
    from google.colab import drive
    drive.mount("/content/drive")

    # Hugging Face login (needed for the gated IndicVoices dataset)
    try:
        from google.colab import userdata
        os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
        print("HF token loaded from Colab secrets")
    except Exception as e:
        print("No HF_TOKEN secret found (", type(e).__name__, ") -> interactive login")
        from huggingface_hub import notebook_login
        notebook_login()

### CONFIG: every setting in one place

Read the comments. The defaults are a sensible first experiment for a free T4.

In [ ]:
# ============================== CONFIG ==============================
QUICK_TEST = False          # True = tiny end-to-end run (~10 min) to check that everything works. Do this first!

# ---- where things go. Drive = survives disconnects. /content = fast but wiped when the session ends.
DRIVE_ROOT   = "/content/drive/MyDrive/moonshine_finetune"
RUN_NAME     = "run1_indicvoices_phone"
OUTPUT_DIR   = f"{DRIVE_ROOT}/{RUN_NAME}"           # checkpoints, logs, results, final model
MY_AUDIO_DIR = f"{DRIVE_ROOT}/sample-audio"          # your own recordings (optional, see Part 2)
WORK_DIR     = "/content/work"                       # downloaded + processed data

BASE_MODEL = "Sanjidh090/moonshine-base-bn"

# ---- training data 1: IndicVoices Bengali (real extempore + conversational speech, CC-BY-4.0, gated)
# The Bengali part has 96 parquet "shards" (~410 MB each). We download only a few.
IV_REPO        = "ai4bharat/IndicVoices"
IV_TRAIN_FILES = [f"bengali/train-{i:05d}-of-00096.parquet" for i in range(10)]
IV_VALID_FILES = ["bengali/valid-00000-of-00001.parquet"]
IV_MAX_TRAIN_HOURS = 60       # stop collecting after this many hours (the T4 can do ~60 h x 3 epochs in a session)
IV_TEXT_COL    = None         # None = auto-detect; set e.g. "normalized" or "verbatim" after looking at the data
IV_KEEP        = None         # e.g. {"task_name": ["conversation"]} to keep only some recording types (see Part 4)

# ---- training data 2: clean Bangladeshi read speech, mixed in so the model does not forget clean audio
SUBAKKO_REPO    = "SUST-CSE-Speech/SUBAK.KO"
SUBAKKO_FILES   = [f"data/train-{i:05d}-of-00047.parquet" for i in range(3)]   # ~4 h per shard
CLEAN_MIX_RATIO = 0.20        # clean hours = 20% of the training set. 0 = no clean mix.

# ---- test data (NEVER trained on): BanSpeech, Bangladeshi broadcast speech in 13 domains
BANSPEECH_REPO  = "SUST-CSE-Speech/banspeech"
BANSPEECH_SPONT = ["political_talkshow", "celebrity_interview", "drama_series"]   # spontaneous, conversational
BANSPEECH_CLEAN = ["television_news", "audio_books"]                               # clean, for "did it forget?"
TEST_CLIPS_PER_SPLIT = 300
DEV_CLIPS            = 300

# ---- filters for training clips
MIN_SEC, MAX_SEC  = 1.0, 20.0  # Moonshine works on short utterances; our inference uses <= 20 s chunks
MAX_LABEL_TOKENS  = 180        # decoder has 194 positions in total
DROP_LATIN        = True       # drop clips whose text has English letters (our rule: English words in Bangla script)
DROP_DIGITS       = True       # drop clips with digits (our rule: numbers written as words, e.g. ষোলো)

# ---- augmentation
PHONE_SIM_PROB = 0.5           # chance that a wide-band training clip is turned into phone audio (re-drawn every epoch)

# ---- training hyper-parameters (explained in Part 1.7)
LEARNING_RATE = 3e-5
NUM_EPOCHS    = 3
MAX_STEPS     = -1             # -1 = use NUM_EPOCHS. A number > 0 overrides epochs.
BATCH_SIZE    = 16
GRAD_ACCUM    = 2              # effective batch = 32
WARMUP        = 0.10
WEIGHT_DECAY  = 0.01
EVAL_STEPS    = 250            # evaluate on dev + save a checkpoint every N steps
SEED          = 42

if QUICK_TEST:
    RUN_NAME = "quick_test"; OUTPUT_DIR = f"{DRIVE_ROOT}/{RUN_NAME}"
    IV_TRAIN_FILES = IV_TRAIN_FILES[:1]; IV_MAX_TRAIN_HOURS = 0.5
    SUBAKKO_FILES = SUBAKKO_FILES[:1]
    TEST_CLIPS_PER_SPLIT = 30; DEV_CLIPS = 40
    MAX_STEPS = 60; EVAL_STEPS = 30
# ====================================================================

In [ ]:
import io, re, json, math, time, glob, random, shutil, tempfile, unicodedata
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import soundfile as sf
import librosa
import jiwer
import pyarrow as pa
import pyarrow.parquet as pq
from scipy.signal import butter, sosfilt, resample_poly
from IPython.display import Audio, display, Markdown

import transformers
from transformers import (AutoTokenizer, AutoModelForSpeechSeq2Seq, GenerationConfig,
                          Seq2SeqTrainer, Seq2SeqTrainingArguments)
from transformers.trainer_utils import get_last_checkpoint
import datasets
from huggingface_hub import hf_hub_download, list_repo_files

SR = 16000
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
for d in (OUTPUT_DIR, WORK_DIR):
    Path(d).mkdir(parents=True, exist_ok=True)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print("transformers", transformers.__version__, "| datasets", datasets.__version__, "| torch", torch.__version__)
print("device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE == "cuda" else "")
print("disk free on /content: %.0f GB" % (shutil.disk_usage(WORK_DIR).free / 1e9))

---
# Part 3: The model, the tokenizer and what a label looks like

One important detail: the model's saved `generation_config.json` has **wrong token ids** (start=1, end=2, pad=2).
The real ids (from `config.json` and the model card) are start=2 `[CLS]`, end=3 `[SEP]`, pad=0 `[PAD]`.
We fix the generation config here, so evaluation during training and the saved model both use the right ids.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
START_ID, EOS_ID, PAD_ID = 2, 3, 0
assert tokenizer.convert_ids_to_tokens([PAD_ID, START_ID, EOS_ID]) == ["[PAD]", "[CLS]", "[SEP]"]

def fixed_generation_config():
    return GenerationConfig(decoder_start_token_id=START_ID, bos_token_id=START_ID, eos_token_id=EOS_ID,
                            pad_token_id=PAD_ID)

def load_model(path=BASE_MODEL):
    m = AutoModelForSpeechSeq2Seq.from_pretrained(path, dtype=torch.float32)
    m.generation_config = fixed_generation_config()
    return m.to(DEVICE)

model = load_model()
print("parameters: %.1f M" % (sum(p.numel() for p in model.parameters()) / 1e6))
print("decoder positions:", model.config.max_position_embeddings)

# How a sentence becomes a training label
s = "আসসালামু আলাইকুম ডাক্তার রেজাউল হক কয়টার সময় বসবেন"
ids = tokenizer(s, add_special_tokens=False)["input_ids"]
print("\ntext   :", s)
print("tokens :", tokenizer.convert_ids_to_tokens(ids))
print("labels :", ids + [EOS_ID], " <- text tokens + [SEP]; NO [CLS] at the front")
print("decoder input (made by the model from the labels):", [START_ID] + ids)

**Why no `[CLS]` in the labels?** The model adds `[CLS]` itself as the first *decoder input*. If the labels
also started with `[CLS]`, the model would learn to write `[CLS]` as its first output word. We checked this
on the original model: labels *without* `[CLS]` give a clearly lower loss (1.78 vs 2.33 on a news clip), so
that is how it was trained.

---
# Part 4: Data

### Helper functions
* `normalize_text`: one cleaning rule for **both** training labels and scoring. Unicode NFC, remove tags like
  `[noise]` / `<laugh>`, remove punctuation (`।`, `,`, `?` …), collapse spaces. If labels and references are
  cleaned differently, WER measures the cleaning, not the model.
* `decode_audio`: bytes from the parquet file → mono float → 16 kHz.
* `phone_sim`: turn a clip into phone-call audio (explained and played in Part 5).

In [ ]:
_TAGS  = re.compile(r"\[[^\]]*\]|<[^>]*>|\{[^}]*\}")
_PUNCT = re.compile(r"[।॥,\.\?!;:\"'‘’“”\-–—\(\)/\\…*_~`|=+#@&%^]")
_ZW    = re.compile(r"[​‌‍﻿]")
_LATIN = re.compile(r"[A-Za-z]")
_DIGIT = re.compile(r"[0-9০-৯]")

def normalize_text(s):
    s = unicodedata.normalize("NFC", s or "")
    s = _TAGS.sub(" ", s)
    s = _ZW.sub("", s)
    s = _PUNCT.sub(" ", s)
    return re.sub(r"\s+", " ", s).strip()

def decode_audio(b):
    # returns (16 kHz mono float32, native sample rate)
    try:
        y, sr = sf.read(io.BytesIO(b), dtype="float32", always_2d=True)
        y = y.mean(axis=1)
    except Exception:
        with tempfile.NamedTemporaryFile(suffix=".bin", delete=False) as f:
            f.write(b); tmp = f.name
        try:
            y, sr = librosa.load(tmp, sr=None, mono=True)
        finally:
            os.remove(tmp)
    if sr != SR:
        y = librosa.resample(y, orig_sr=sr, target_sr=SR)
    return y.astype(np.float32), int(sr)

_SOS_PHONE = butter(4, [300, 3400], btype="bandpass", fs=8000, output="sos")

def phone_sim(y, rng):
    # 16 kHz clip -> phone-line version (still returned at 16 kHz, like our real calls after upsampling)
    y8 = resample_poly(y, 1, 2)                              # 16 kHz -> 8 kHz: everything above 4 kHz is gone
    y8 = sosfilt(_SOS_PHONE, y8)                             # phone band 300-3400 Hz
    y8 = y8 / (np.abs(y8).max() + 1e-9) * rng.uniform(0.3, 0.9)   # random level
    mu = 255.0                                               # G.711 mu-law codec: compress, quantise to 8 bit, expand
    c = np.sign(y8) * np.log1p(mu * np.abs(y8)) / np.log1p(mu)
    c = np.round((c + 1) / 2 * mu) / mu * 2 - 1
    y8 = np.sign(c) * ((1 + mu) ** np.abs(c) - 1) / mu
    snr_db = rng.uniform(15, 35)                             # line / room noise
    p = np.mean(y8 ** 2) + 1e-12
    y8 = y8 + rng.normal(0, np.sqrt(p / 10 ** (snr_db / 10)), len(y8))
    return np.clip(resample_poly(y8, 2, 1), -1, 1).astype(np.float32)

def download(repo, files):
    # downloads dataset files to WORK_DIR/raw (skips files already there)
    out = []
    for f in files:
        try:
            out.append(hf_hub_download(repo_id=repo, filename=f, repo_type="dataset",
                                       local_dir=f"{WORK_DIR}/raw/{repo.replace('/', '__')}"))
        except Exception as e:
            msg = str(e)
            if "gated" in msg.lower() or "401" in msg or "403" in msg:
                raise RuntimeError(f"No access to {repo}. Open https://huggingface.co/datasets/{repo}, accept the "
                                   f"conditions with the same account as your HF_TOKEN, then re-run.") from e
            raise
        print("ok", f, "%.0f MB" % (os.path.getsize(out[-1]) / 1e6))
    return out

def find_audio_col(schema):
    for f in schema:
        if pa.types.is_struct(f.type) and "bytes" in [c.name for c in f.type]:
            return f.name
    raise ValueError(f"no audio column (struct with 'bytes') in {schema.names}")

def string_cols(schema):
    return [f.name for f in schema if pa.types.is_string(f.type) or pa.types.is_large_string(f.type)]

### 4.1 Download and **look at** IndicVoices Bengali

Always look at the data before training. Things to check below:
* **Which column holds the transcript?** IndicVoices may have more than one text version (e.g. *verbatim* =
  exactly as spoken, *normalized* = cleaned up). We want **as spoken, without tags**. Auto-detection picks the
  first of `text, normalized, verbatim, transcription, transcript, sentence`. If the samples printed below show
  another column is better, set `IV_TEXT_COL` in CONFIG and re-run from here.
* **Recording types:** short-list columns (few distinct values), e.g. task/scenario/gender/district. If there
  is a column that marks *conversation* recordings, you can restrict training to it with `IV_KEEP`.
* **Native sample rate:** IndicVoices recorded the two-person conversations **over a real 8 kHz telephone
  bridge**. Those clips show up as `8000` below. They are the closest thing to our call-center audio in any
  open Bangla dataset.

In [ ]:
iv_train_paths = download(IV_REPO, IV_TRAIN_FILES)
iv_valid_paths = download(IV_REPO, IV_VALID_FILES)

schema = pq.ParquetFile(iv_train_paths[0]).schema_arrow
IV_AUDIO_COL = find_audio_col(schema)
str_cols = string_cols(schema)
print("all columns   :", schema.names)
print("audio column  :", IV_AUDIO_COL)
print("string columns:", str_cols)

if IV_TEXT_COL is None:
    IV_TEXT_COL = next((c for c in ["text", "normalized", "verbatim", "transcription", "transcript", "sentence"]
                        if c in str_cols), None)
assert IV_TEXT_COL, "Could not auto-detect the text column; set IV_TEXT_COL in CONFIG"
print("text column   :", IV_TEXT_COL, "  <- change IV_TEXT_COL in CONFIG if the samples show a better one")

peek = pd.concat([b.to_pandas() for _, b in zip(range(4), pq.ParquetFile(iv_train_paths[0]).iter_batches(500, columns=str_cols))])
print("\n--- 3 example rows (all text columns) ---")
for _, r in peek.head(3).iterrows():
    for c in str_cols:
        print(f"{c:>22}: {str(r[c])[:150]}")
    print()

print("--- short-list columns (candidates for IV_KEEP) ---")
for c in str_cols:
    vc = peek[c].value_counts()
    if 1 < len(vc) <= 30:
        print(f"\n[{c}]\n{vc.to_string()}")

In [ ]:
# native sample rates + durations of the first 300 clips, and listen to 3 of them
pf = pq.ParquetFile(iv_train_paths[0])
rows = next(pf.iter_batches(300, columns=[IV_AUDIO_COL, IV_TEXT_COL])).to_pylist()
srs, durs = Counter(), []
for r in rows:
    info = sf.info(io.BytesIO(r[IV_AUDIO_COL]["bytes"])) if r[IV_AUDIO_COL]["bytes"] else None
    if info:
        srs[info.samplerate] += 1; durs.append(info.duration)
print("native sample rates:", dict(srs), " (8000 = telephone recordings)")
print("duration: mean %.1f s, max %.1f s" % (np.mean(durs), np.max(durs)))
for r in rows[:3]:
    y, sr = decode_audio(r[IV_AUDIO_COL]["bytes"])
    print(f"\n[{sr} Hz, {len(y)/SR:.1f} s] {r[IV_TEXT_COL]}")
    display(Audio(y, rate=SR))

### 4.2 Build the training set

For every clip: decode → 16 kHz → clean the text → check filters → tokenize → store as 16-bit audio on disk
(an Arrow dataset, so it does not have to fit in RAM). The filters drop clips that would teach the model
something wrong:

| Filter | Why |
|---|---|
| empty text, < 1 s or > 20 s | too short to be useful / longer than what we transcribe at once |
| > 180 label tokens | the decoder has only 194 positions |
| > 6 or < 0.3 words per second (clips > 3 s) | almost always a wrong or partial transcript |
| English letters / digits in the text | against our transcription rules (English words and numbers written in Bangla) |

In [ ]:
try:
    from datasets import List as _List
except ImportError:
    from datasets import Sequence as _List
from datasets import Dataset, Features, Value, concatenate_datasets

FEATURES = Features({
    "audio": _List(Value("int16")), "text": Value("string"), "labels": _List(Value("int32")),
    "length": Value("int32"), "native_sr": Value("int32"), "source": Value("string"),
    "category": Value("string"), "augment": Value("int8"),
})

def check_clip(text, dur):
    # returns (clean text, label ids) or (None, reason)
    t = normalize_text(text)
    if not t:                              return None, "empty text"
    if dur < MIN_SEC:                      return None, "too short"
    if dur > MAX_SEC:                      return None, "too long"
    if DROP_LATIN and _LATIN.search(t):    return None, "latin letters"
    if DROP_DIGITS and _DIGIT.search(t):   return None, "digits"
    wps = len(t.split()) / dur
    if dur > 3 and (wps > 6 or wps < 0.3): return None, "words/sec implausible"
    ids = tokenizer(t, add_special_tokens=False)["input_ids"] + [EOS_ID]
    if len(ids) > MAX_LABEL_TOKENS:        return None, "too many tokens"
    if tokenizer.unk_token_id in ids:      return None, "unknown token"
    return t, ids

def iter_clips(paths, audio_col, text_col, keep=None, category_col=None):
    # yields (16 kHz float audio, native sr, raw text, category) from parquet files, one row group at a time
    cols = list(dict.fromkeys([audio_col, text_col] + ([category_col] if category_col else []) + list(keep or {})))
    for p in paths:
        pf = pq.ParquetFile(p)
        for g in range(pf.num_row_groups):
            for r in pf.read_row_group(g, columns=cols).to_pylist():
                if keep and any(r.get(k) not in v for k, v in keep.items()):
                    continue
                b = (r[audio_col] or {}).get("bytes")
                if not b:
                    continue
                y, sr = decode_audio(b)
                yield y, sr, r[text_col], (str(r.get(category_col)) if category_col else "")

DROPS = Counter()

def gen_rows(paths, audio_col, text_col, source, max_hours, keep=None, category_col=None, augment=1, cfg=""):
    total = 0.0
    for y, sr, text, cat in iter_clips(paths, audio_col, text_col, keep, category_col):
        t, ids = check_clip(text, len(y) / SR)
        if t is None:
            DROPS[f"{source}: {ids}"] += 1; continue
        yield {"audio": np.clip(y * 32767, -32768, 32767).astype(np.int16), "text": t, "labels": ids,
               "length": len(y), "native_sr": sr, "source": source, "category": cat, "augment": augment}
        total += len(y) / SR
        if total >= max_hours * 3600:
            return

def build(name, **kw):
    # cfg is part of the cache key: changing a filter setting rebuilds instead of reusing an old cache
    kw["cfg"] = repr((MIN_SEC, MAX_SEC, MAX_LABEL_TOKENS, DROP_LATIN, DROP_DIGITS))
    return Dataset.from_generator(gen_rows, gen_kwargs=kw, features=FEATURES, cache_dir=f"{WORK_DIR}/cache/{name}")

def summary(ds, by="source"):
    df = pd.DataFrame({by: ds[by], "sec": np.array(ds["length"]) / SR})
    out = df.groupby(by)["sec"].agg(clips="count", hours=lambda s: s.sum() / 3600, mean_sec="mean")
    return out.round(2)

# a short-list column to report on (e.g. task/scenario), if one exists
IV_CATEGORY_COL = next((c for c in ["task_name", "scenario", "task", "audio_type", "domain"] if c in str_cols), None)
print("category column:", IV_CATEGORY_COL)

t0 = time.time()
DROPS.clear()
iv_train = build("iv_train", paths=iv_train_paths, audio_col=IV_AUDIO_COL, text_col=IV_TEXT_COL,
                 source="indicvoices", max_hours=IV_MAX_TRAIN_HOURS, keep=IV_KEEP, category_col=IV_CATEGORY_COL)
print("built in %.1f min" % ((time.time() - t0) / 60))
print(summary(iv_train))
if IV_CATEGORY_COL: print(summary(iv_train, "category"))
print(summary(iv_train, "native_sr"))
print("dropped:", dict(DROPS) if DROPS else "(loaded from cache)")

### 4.3 Clean-speech mix (SUBAK.KO)

Why: the model was trained on clean-ish YouTube audio and does well on it. If we only train on new
conversational / phone audio, it can slowly **forget** how to handle clean audio. Mixing in ~20% clean read speech
from Bangladeshi speakers (SUBAK.KO) is the standard protection ("rehearsal"). These clips are not
phone-simulated (`augment=0` for the phone sim; they only get a small volume change).

*Why not Lipi-Ghor, the model's original training data?* That would be the ideal rehearsal set, but it is
gated, 50 GB, and has an unclear file layout. SUBAK.KO does the same job here. If the clean test set gets
worse after training, adding a slice of Lipi-Ghor is the next step (see README).

In [ ]:
clean_hours = 0.0
if CLEAN_MIX_RATIO > 0:
    iv_hours = sum(iv_train["length"]) / SR / 3600
    clean_hours = iv_hours * CLEAN_MIX_RATIO / (1 - CLEAN_MIX_RATIO)
    sk_paths = download(SUBAKKO_REPO, SUBAKKO_FILES)
    sk_schema = pq.ParquetFile(sk_paths[0]).schema_arrow
    DROPS.clear()
    clean_train = build("subakko_train", paths=sk_paths, audio_col=find_audio_col(sk_schema),
                        text_col="transcription", source="subakko_clean", max_hours=clean_hours, augment=2)
    print(summary(clean_train)); print("dropped:", dict(DROPS) if DROPS else "(cache)")
    if sum(clean_train["length"]) / SR / 3600 < 0.9 * clean_hours:
        print("NOTE: fewer clean hours than the target; add more SUBAKKO_FILES for an exact ratio")
    train_ds = concatenate_datasets([iv_train, clean_train])
else:
    train_ds = iv_train
train_ds = train_ds.shuffle(seed=SEED).with_format("numpy")   # numpy format: fast audio loading
print("\nTRAIN:", len(train_ds), "clips, %.1f hours" % (sum(train_ds["length"]) / SR / 3600))

### 4.4 Dev and test sets

| Name | From | Used for |
|---|---|---|
| `dev_iv_phone` | IndicVoices *valid* split, phone-simulated | choosing the best checkpoint during training |
| `test_iv_phone` | other clips of IndicVoices *valid*, phone-simulated | in-domain final score |
| `test_bs_spont_phone` | BanSpeech talk-show, interview, drama, **phone-simulated** | **main score**: closest open proxy for our calls (Bangladeshi, spontaneous, phone sound) |
| `test_bs_spont_clean` | the same clips without phone simulation | how much of the gain is "phone" vs. "conversation" |
| `test_bs_clean` | BanSpeech TV news + audiobooks, clean | **forgetting check**: must not get worse |

The phone simulation for dev/test uses a fixed random seed, so the exact same audio is used before and after
training. Test clips are kept in memory as float arrays, since they are small.

In [ ]:
def load_rows(paths, audio_col, text_col):
    # all clips that pass the filters: list of (16 kHz float audio, clean text, native sr)
    rows = []
    for y, sr, text, _ in iter_clips(paths, audio_col, text_col):
        t, ids = check_clip(text, len(y) / SR)
        if t is not None:
            rows.append((y, t, sr))
    return rows

def sample_rows(rows, n, seed, phone, exclude=()):
    # n random clips; phone=True -> phone-simulate (fixed seed per clip, so it is identical every run)
    rng, exclude = np.random.default_rng(seed), set(exclude)
    idx = [int(i) for i in rng.permutation(len(rows)) if int(i) not in exclude][:n]
    out = []
    for i in idx:
        y, t, sr = rows[i]
        if phone and sr > 8000:
            y = phone_sim(y, np.random.default_rng(seed * 100003 + int(i)))
        out.append((y, t, sr))
    return out, idx

# IndicVoices valid -> dev + in-domain test (disjoint clips)
iv_valid_all = load_rows(iv_valid_paths, IV_AUDIO_COL, IV_TEXT_COL)
print("IndicVoices valid: %d usable clips" % len(iv_valid_all))
dev_rows, dev_idx = sample_rows(iv_valid_all, DEV_CLIPS, seed=1, phone=True)
ivt_rows, _ = sample_rows(iv_valid_all, TEST_CLIPS_PER_SPLIT, seed=1, phone=True, exclude=dev_idx)
del iv_valid_all

# BanSpeech (file names contain a hash, so look them up)
bs_files = {f.split("/")[-1].split("-")[0]: f for f in list_repo_files(BANSPEECH_REPO, repo_type="dataset")
            if f.startswith("data/") and f.endswith(".parquet")}
_bs_cache = {}
def banspeech(splits, phone, seed=7):
    out = []
    for k, s in enumerate(splits):
        if s not in _bs_cache:
            _bs_cache[s] = load_rows(download(BANSPEECH_REPO, [bs_files[s]]), "audio", "transcription")
        rows, _ = sample_rows(_bs_cache[s], TEST_CLIPS_PER_SPLIT, seed=seed + k, phone=phone)
        out += rows
    return out

TEST_SETS = {
    "test_iv_phone":       ivt_rows,
    "test_bs_spont_phone": banspeech(BANSPEECH_SPONT, phone=True),
    "test_bs_spont_clean": banspeech(BANSPEECH_SPONT, phone=False),
    "test_bs_clean":       banspeech(BANSPEECH_CLEAN, phone=False),
}
_bs_cache.clear()
for k, v in TEST_SETS.items():
    print(f"{k:22s} {len(v):5d} clips  {sum(len(r[0]) for r in v) / SR / 3600:.2f} h")

def rows_to_dataset(rows, source):
    return Dataset.from_dict({
        "audio": [np.clip(y * 32767, -32768, 32767).astype(np.int16) for y, _, _ in rows],
        "text": [t for _, t, _ in rows],
        "labels": [tokenizer(t, add_special_tokens=False)["input_ids"] + [EOS_ID] for _, t, _ in rows],
        "length": [len(y) for y, _, _ in rows], "native_sr": [sr for _, _, sr in rows],
        "source": [source] * len(rows), "category": [""] * len(rows), "augment": [0] * len(rows),
    }, features=FEATURES)

dev_ds = rows_to_dataset(dev_rows, "dev_iv_phone").with_format("numpy")
print("DEV:", len(dev_ds), "clips")

---
# Part 5: Listen to the phone simulation

Compare the same clip clean and phone-simulated, and the spectrograms: the phone version has **nothing above
~3.4 kHz** (dark top half), like your real call recordings. If you have your own call recording on Drive,
it's shown too, so you can check the simulation is realistic.

In [ ]:
import matplotlib.pyplot as plt

y_clean, t_clean, _ = TEST_SETS["test_bs_spont_clean"][0]
y_phone, _, _ = TEST_SETS["test_bs_spont_phone"][0]
print("text:", t_clean)
print("clean:"); display(Audio(y_clean, rate=SR))
print("phone-simulated:"); display(Audio(y_phone, rate=SR))

real = sorted(glob.glob(f"{MY_AUDIO_DIR}/*call*")) if os.path.isdir(MY_AUDIO_DIR) else []
panels = [("clean", y_clean), ("phone-simulated", y_phone)]
if real:
    y_real, _ = librosa.load(real[0], sr=SR, offset=10, duration=6)
    panels.append(("real call: " + Path(real[0]).name, y_real))
    print("real call:"); display(Audio(y_real, rate=SR))

fig, axes = plt.subplots(1, len(panels), figsize=(5 * len(panels), 3.2))
for ax, (title, y) in zip(np.atleast_1d(axes), panels):
    S = librosa.amplitude_to_db(np.abs(librosa.stft(y, n_fft=512)), ref=np.max)
    ax.imshow(S, origin="lower", aspect="auto", cmap="magma", extent=[0, len(y) / SR, 0, SR / 2])
    ax.axhline(3400, color="cyan", lw=0.8, ls="--")
    ax.set_title(title); ax.set_xlabel("seconds"); ax.set_ylabel("Hz")
plt.tight_layout(); plt.show()

---
# Part 6: Baseline, the original model's scores **before** training

Without a baseline you can't tell whether training helped. We decode exactly like the production script
`moonshine_bn.py` does: **beam 5, `no_repeat_ngram_size=3`, `repetition_penalty=1.2`**, max tokens tied to the
audio length. (During training we use faster greedy decoding on the dev set; the final comparison uses these
production settings.)

In [ ]:
@torch.no_grad()
def transcribe(m, arrays, beams=5, batch_size=16):
    m.eval()
    order = np.argsort([-len(a) for a in arrays])          # longest first: less padding per batch
    out = [None] * len(arrays)
    for s in range(0, len(order), batch_size):
        idx = order[s:s + batch_size]
        batch = [arrays[i] for i in idx]
        n = max(len(a) for a in batch); n += (-n) % 320
        x = np.zeros((len(batch), n), np.float32); am = np.zeros((len(batch), n), np.int64)
        for j, a in enumerate(batch):
            x[j, :len(a)] = a; am[j, :len(a)] = 1
        max_new = min(int(n / SR * 8) + 16, 190)
        with torch.autocast("cuda", dtype=torch.float16, enabled=(DEVICE == "cuda")):
            ids = m.generate(torch.from_numpy(x).to(DEVICE), attention_mask=torch.from_numpy(am).to(DEVICE),
                             max_new_tokens=max_new, num_beams=beams, no_repeat_ngram_size=3,
                             repetition_penalty=1.2, decoder_start_token_id=START_ID,
                             eos_token_id=EOS_ID, pad_token_id=PAD_ID)
        for j, i in enumerate(idx):
            out[i] = normalize_text(tokenizer.decode(ids[j], skip_special_tokens=True))
    return out

def score(refs, hyps):
    pairs = [(r, h if h else "<empty>") for r, h in zip(refs, hyps) if r]
    r, h = zip(*pairs)
    return jiwer.wer(list(r), list(h)), jiwer.cer(list(r), list(h))

def evaluate_all(m, label, beams=5):
    rows, hyps_all = [], {}
    for name, data in TEST_SETS.items():
        t0 = time.time()
        hyps = transcribe(m, [y for y, _, _ in data], beams=beams)
        w, c = score([t for _, t, _ in data], hyps)
        hyps_all[name] = hyps
        rows.append({"model": label, "test_set": name, "clips": len(data), "WER": round(w, 4), "CER": round(c, 4),
                     "sec": round(time.time() - t0, 1)})
        print(f"{label:10s} {name:22s} WER {w:.3f}  CER {c:.3f}  ({time.time() - t0:.0f}s)")
    return pd.DataFrame(rows), hyps_all

baseline_df, baseline_hyps = evaluate_all(model, "baseline")
baseline_df.to_csv(f"{OUTPUT_DIR}/baseline_scores.csv", index=False)
display(baseline_df)

**What to expect:** `test_bs_spont_phone` should be clearly worse than `test_bs_spont_clean`. That gap is the
"phone problem" we are trying to shrink. Look at a few examples to get a feeling for the errors:

In [ ]:
for i in range(min(5, len(TEST_SETS["test_bs_spont_phone"]))):
    print("REF :", TEST_SETS["test_bs_spont_phone"][i][1])
    print("HYP :", baseline_hyps["test_bs_spont_phone"][i]); print()

---
# Part 7: Training

### 7.1 The data collator: how one training batch is made

Each step, the trainer takes 16 clips and the **collator**:
1. converts the stored 16-bit audio back to floats,
2. **augments** IndicVoices wide-band clips: with probability `PHONE_SIM_PROB` they become phone audio
   (a new random version every epoch, so the model never sees exactly the same audio twice).
   Clips that are already 8 kHz telephone recordings are left as they are. Clean SUBAK.KO clips get only a
   small volume change,
3. pads all clips to the same length (a multiple of 320 samples, which Moonshine's conv stem needs) and builds
   an **attention mask** (1 = real audio, 0 = padding) so the model ignores the padding,
4. pads the labels with `-100`. PyTorch's loss ignores `-100`, so padding doesn't count.

In [ ]:
class SpeechCollator:
    def __init__(self, seed=SEED):
        self.seed, self.pid, self.rng = seed, None, None

    def __call__(self, feats):
        if self.pid != os.getpid():                     # every dataloader worker gets its own random stream
            self.pid = os.getpid(); self.rng = np.random.default_rng(self.seed + self.pid + int(time.time()))
        waves = []
        for f in feats:
            y = np.asarray(f["audio"], dtype=np.float32) / 32768.0
            aug = int(f["augment"])
            if aug == 1 and int(f["native_sr"]) > 8000 and self.rng.random() < PHONE_SIM_PROB:
                y = phone_sim(y, self.rng)
            elif aug in (1, 2):
                y = np.clip(y * self.rng.uniform(0.6, 1.4), -1, 1)
            waves.append(y)
        n = max(len(w) for w in waves); n += (-n) % 320
        x = np.zeros((len(waves), n), np.float32); am = np.zeros((len(waves), n), np.int64)
        for i, w in enumerate(waves):
            x[i, :len(w)] = w; am[i, :len(w)] = 1
        L = max(len(f["labels"]) for f in feats)
        lab = np.full((len(feats), L), -100, np.int64)
        for i, f in enumerate(feats):
            lab[i, :len(f["labels"])] = f["labels"]
        return {"input_values": torch.from_numpy(x), "attention_mask": torch.from_numpy(am),
                "labels": torch.from_numpy(lab)}

collator = SpeechCollator()
b = collator([train_ds[i] for i in range(4)])
print({k: tuple(v.shape) for k, v in b.items()})
print("labels[0]:", b["labels"][0][:12].tolist(), "...")

### 7.2 Metrics during training and the trainer settings

Every `EVAL_STEPS` steps the trainer **generates** text for the dev set (greedy, fast), computes WER/CER, and
saves a checkpoint to Drive. At the end it reloads the checkpoint with the **lowest dev WER**, which is not
necessarily the last one.

If Colab disconnects: re-run all cells. The trainer finds the last checkpoint on Drive and **continues** from
there (data preparation has to run again, because `/content` is wiped).

In [ ]:
def compute_metrics(pred):
    p = np.where(pred.predictions < 0, PAD_ID, pred.predictions)
    l = np.where(pred.label_ids < 0, PAD_ID, pred.label_ids)
    hyps = [normalize_text(s) for s in tokenizer.batch_decode(p, skip_special_tokens=True)]
    refs = [normalize_text(s) for s in tokenizer.batch_decode(l, skip_special_tokens=True)]
    w, c = score(refs, hyps)
    return {"wer": w, "cer": c}

CKPT_DIR = f"{OUTPUT_DIR}/checkpoints"
steps_per_epoch = math.ceil(len(train_ds) / (BATCH_SIZE * GRAD_ACCUM))
total_steps = MAX_STEPS if MAX_STEPS > 0 else steps_per_epoch * NUM_EPOCHS
print(f"{len(train_ds)} clips -> {steps_per_epoch} updates per epoch -> {total_steps} updates in total")

args = Seq2SeqTrainingArguments(
    output_dir=CKPT_DIR,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="linear",
    warmup_steps=WARMUP,
    weight_decay=WEIGHT_DECAY,
    max_grad_norm=1.0,
    num_train_epochs=NUM_EPOCHS,
    max_steps=MAX_STEPS,
    fp16=(DEVICE == "cuda"),
    eval_strategy="steps", eval_steps=EVAL_STEPS,
    save_strategy="steps", save_steps=EVAL_STEPS, save_total_limit=2,
    load_best_model_at_end=True, metric_for_best_model="wer", greater_is_better=False,
    predict_with_generate=True, generation_max_length=160, generation_num_beams=1,
    logging_steps=25, report_to="none",
    remove_unused_columns=False,
    train_sampling_strategy="group_by_length", length_column_name="length",
    dataloader_num_workers=2 if DEVICE == "cuda" else 0,
    seed=SEED,
)

model = load_model()   # fresh copy of the original model (the baseline run did not change it, but be explicit)
trainer = Seq2SeqTrainer(model=model, args=args, train_dataset=train_ds, eval_dataset=dev_ds,
                         data_collator=collator, compute_metrics=compute_metrics, processing_class=tokenizer)

### 7.3 Train

What to watch in the table that appears:
* **Training Loss** should go down, fast at first, then slowly.
* **Validation Wer** should go down. If it goes **up** for two evaluations in a row while training loss
  keeps falling, the model is overfitting: the best checkpoint is kept anyway, and next time use fewer
  epochs or a lower learning rate.
* `nan` loss → lower `LEARNING_RATE` (e.g. `1e-5`) and restart.
* "CUDA out of memory" → `BATCH_SIZE = 8`, `GRAD_ACCUM = 4` (same effective batch).

In [ ]:
last = get_last_checkpoint(CKPT_DIR) if os.path.isdir(CKPT_DIR) else None
print("resuming from:", last) if last else print("starting fresh")
t0 = time.time()
train_result = trainer.train(resume_from_checkpoint=last)
print("training time: %.1f min" % ((time.time() - t0) / 60))
print("best checkpoint:", trainer.state.best_model_checkpoint, "| best dev WER:", trainer.state.best_metric)

In [ ]:
# Learning curves
hist = pd.DataFrame(trainer.state.log_history)
hist.to_csv(f"{OUTPUT_DIR}/log_history.csv", index=False)
fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
tr = hist.dropna(subset=["loss"]) if "loss" in hist else pd.DataFrame()
if len(tr): ax[0].plot(tr["step"], tr["loss"], label="train loss")
ev = hist.dropna(subset=["eval_loss"]) if "eval_loss" in hist else pd.DataFrame()
if len(ev):
    ax[0].plot(ev["step"], ev["eval_loss"], "o-", label="dev loss")
    ax[1].plot(ev["step"], ev["eval_wer"], "o-", label="dev WER"); ax[1].plot(ev["step"], ev["eval_cer"], "o-", label="dev CER")
for a in ax: a.set_xlabel("step"); a.legend(); a.grid(alpha=.3)
ax[0].set_title("loss"); ax[1].set_title("dev error rate (greedy)")
plt.tight_layout(); plt.savefig(f"{OUTPUT_DIR}/curves.png", dpi=120); plt.show()

---
# Part 8: Evaluate, before vs. after

Same test sets, same decoding settings (beam 5, production settings) as the baseline. **How to read it:**

| Result | Meaning |
|---|---|
| `test_bs_spont_phone` down clearly (e.g. ≥ 3 WER points) | phone + conversational training helps → worth labelling our own calls |
| `test_bs_spont_clean` also down | it also learned conversational speech, not only the phone sound |
| `test_bs_clean` up by more than ~1–2 points | forgetting → raise `CLEAN_MIX_RATIO`, lower LR, or fewer epochs |
| everything roughly the same | the open data is too different from our audio; go straight to our own call data |

The WER differences on a few hundred short clips have noise of roughly ±1 point, so don't read much into
smaller changes.

In [ ]:
model = trainer.model
model.generation_config = fixed_generation_config()
ft_df, ft_hyps = evaluate_all(model, "finetuned")
both = pd.concat([baseline_df, ft_df])
cmp = both.pivot(index="test_set", columns="model", values=["WER", "CER"])
cmp[("WER", "change")] = cmp[("WER", "finetuned")] - cmp[("WER", "baseline")]
cmp[("CER", "change")] = cmp[("CER", "finetuned")] - cmp[("CER", "baseline")]
cmp = cmp.round(4)
both.to_csv(f"{OUTPUT_DIR}/final_scores.csv", index=False)
display(cmp)

In [ ]:
# Side by side on the main test set: where did it improve, where did it get worse?
name = "test_bs_spont_phone"
for i in range(min(8, len(TEST_SETS[name]))):
    print("REF       :", TEST_SETS[name][i][1])
    print("baseline  :", baseline_hyps[name][i])
    print("finetuned :", ft_hyps[name][i]); print()

### 8.1 Your own recordings

Transcribes every audio file in `MY_AUDIO_DIR` with both models, in 20-second chunks like
`moonshine_base_bn/test_model.py`. If a file `<name>.txt` with the correct transcript is next to the audio,
WER/CER are computed too. Writing a reference for your two call recordings (~95 s of audio) takes ~15
minutes and turns "looks better" into a number.

In [ ]:
def transcribe_file(m, path, chunk_sec=20):
    y, _ = librosa.load(path, sr=SR, mono=True)
    size = int(chunk_sec * SR)
    chunks = [y[i:i + size] for i in range(0, len(y), size)] or [y]
    chunks = [c for c in chunks if len(c) >= 1600]
    return " ".join(h for h in transcribe(m, chunks, batch_size=4) if h)

base_model = load_model()
report = [f"# Own recordings: baseline vs {RUN_NAME}\n"]
files = sorted(p for p in glob.glob(f"{MY_AUDIO_DIR}/*") if Path(p).suffix.lower() in
               {".wav", ".mp3", ".flac", ".ogg", ".m4a"}) if os.path.isdir(MY_AUDIO_DIR) else []
if not files:
    print("No files in", MY_AUDIO_DIR, "- upload your sample-audio folder there to use this step.")
for p in files:
    b, f = transcribe_file(base_model, p), transcribe_file(model, p)
    report.append(f"## {Path(p).name}\n\n**baseline:** {b}\n\n**finetuned:** {f}\n")
    ref_path = Path(p).with_suffix(".txt")
    if ref_path.exists():
        ref = normalize_text(ref_path.read_text(encoding="utf-8"))
        (wb, cb), (wf, cf) = score([ref], [b]), score([ref], [f])
        report.append(f"WER {wb:.3f} → {wf:.3f} | CER {cb:.3f} → {cf:.3f}\n")
if files:
    Path(f"{OUTPUT_DIR}/own_recordings.md").write_text("\n".join(report), encoding="utf-8")
    display(Markdown("\n".join(report)))
del base_model

---
# Part 9: Save the model and use it on your PC

The final model (best checkpoint) is saved to Drive as `OUTPUT_DIR/final_model` (~250 MB) together with the
tokenizer, the **fixed** `generation_config.json`, and a `training_info.json` describing this run. A zip is also
made for downloading.

**On your PC:**
1. Download `final_model.zip` from Drive and unzip it into
   `Fine-tune-moonshine/models/<run name>/` (this folder is git-ignored).
2. Run `python Fine-tune-moonshine/test_finetuned.py --model Fine-tune-moonshine/models/<run name>`.
   It transcribes `sample-audio/` with the original and the fine-tuned model on CPU and writes a side-by-side
   report, including speed (RTF) and WER/CER where reference `.txt` files exist.
3. The folder is a normal Hugging Face model folder, so `MoonshineBN("<folder>")` from
   `moonshine_base_bn/moonshine_bn.py` loads it directly.

In [ ]:
FINAL_DIR = f"{OUTPUT_DIR}/final_model"
trainer.save_model(FINAL_DIR)
tokenizer.save_pretrained(FINAL_DIR)
fixed_generation_config().save_pretrained(FINAL_DIR)

info = {
    "base_model": BASE_MODEL, "run_name": RUN_NAME, "quick_test": QUICK_TEST,
    "train_clips": len(train_ds), "train_hours": round(sum(train_ds["length"]) / SR / 3600, 2),
    "train_by_source": summary(train_ds).reset_index().to_dict("records"),
    "iv_text_col": IV_TEXT_COL, "iv_keep": IV_KEEP, "iv_files": IV_TRAIN_FILES, "subakko_files": SUBAKKO_FILES,
    "clean_mix_ratio": CLEAN_MIX_RATIO, "phone_sim_prob": PHONE_SIM_PROB,
    "learning_rate": LEARNING_RATE, "epochs": NUM_EPOCHS, "max_steps": MAX_STEPS, "batch": BATCH_SIZE,
    "grad_accum": GRAD_ACCUM, "warmup": WARMUP, "best_checkpoint": trainer.state.best_model_checkpoint,
    "best_dev_wer_greedy": trainer.state.best_metric,
    "scores": both.to_dict("records"),
    "versions": {"transformers": transformers.__version__, "torch": torch.__version__},
}
Path(f"{FINAL_DIR}/training_info.json").write_text(json.dumps(info, ensure_ascii=False, indent=2), encoding="utf-8")
shutil.make_archive(f"{OUTPUT_DIR}/final_model", "zip", FINAL_DIR)
print("saved:", FINAL_DIR)
print("zip  :", f"{OUTPUT_DIR}/final_model.zip", "%.0f MB" % (os.path.getsize(f"{OUTPUT_DIR}/final_model.zip") / 1e6))

---
# What next?

* **Write down the result table** (it is also in `OUTPUT_DIR/final_scores.csv`) in the README's experiment log.
* **If the phone test improved:** this shows the approach works. Next real step: our own labelled call
  segments (`MOONSHINE_FINETUNE_PLAN.md`, steps 1–4). Fine-tune *this* model or the original one on them.
* **Experiments worth trying, one change at a time:**
  1. `IV_KEEP` = only conversation recordings (if the data has such a column) vs. everything
  2. `PHONE_SIM_PROB` 0.5 → 0.8
  3. more data: `IV_TRAIN_FILES` 10 → 20 shards and `IV_MAX_TRAIN_HOURS` 60 → 120 (needs ~2× time)
  4. `LEARNING_RATE` 3e-5 → 1e-5 if the clean test got worse, or 5e-5 if nothing changed
* Free Colab gives a limited number of GPU hours per day/week. If a run is cut off, re-run all cells: training
  resumes from the last checkpoint on Drive.